1. Import **libraries**

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

2. Load the raw dataset

In [ ]:
if os.path.exists("aqi.csv"):
    raw = pd.read_csv("aqi.csv")
else:
    raw = pd.read_excel("aqi.xlsx")
raw.head()

,date,state,area,number_of_monitoring_stations,prominent_pollutants,aqi_value,air_quality_status,unit,note
0,30-04-2025,Maharashtra,Amravati,2,PM10,78,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
1,30-04-2025,Bihar,Purnia,1,CO,56,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
2,30-04-2025,Madhya Pradesh,Katni,1,O3,98,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
3,30-04-2025,Chhattisgarh,Tumidih,1,PM10,103,Moderate,number_of_monitoring_stations in Absolute Numb...,NaN
4,30-04-2025,Assam,Byrnihat,1,PM2.5,61,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN


3. Size of the dataset

In [ ]:
print("Rows, columns:", raw.shape)

Rows, columns: (235785, 9)


4. Data types

In [ ]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235785 entries, 0 to 235784
Data columns (total 9 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   date                           235785 non-null  object 
 1   state                          235785 non-null  object 
 2   area                           235785 non-null  object 
 3   number_of_monitoring_stations  235785 non-null  int64  
 4   prominent_pollutants           235785 non-null  object 
 5   aqi_value                      235785 non-null  int64  
 6   air_quality_status             235785 non-null  object 
 7   unit                           235785 non-null  object 
 8   note                           0 non-null       float64
dtypes: float64(1), int64(2), object(6)
memory usage: 16.2+ MB


 5. Missing values

In [ ]:
raw.isnull().sum()

,0
date,0
state,0
area,0
number_of_monitoring_stations,0
prominent_pollutants,0
aqi_value,0
air_quality_status,0
unit,0
note,235785


6. Summary of AQI values

In [ ]:
print(raw["state"].nunique(), "states")
print(raw["area"].nunique(), "areas")
print(raw["unit"].nunique(), "unique value in 'unit'")
print()
raw["air_quality_status"].value_counts()

32 states
291 areas
1 unique value in 'unit'



,count
air_quality_status,
Satisfactory,88897
Moderate,77537
Good,41971
Poor,21154
Very Poor,5671
Severe,555


8. Duplicates, date range and repeated areas

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())

dates = pd.to_datetime(raw["date"], dayfirst=True)
print("Date range:", dates.min().date(), "to", dates.max().date())
print("Different days in the data:", dates.nunique())

states_per_area = raw.groupby("area")["state"].nunique()
print("Area names found in more than one state:")
print(states_per_area[states_per_area > 1])

Duplicate rows: 0
Date range: 2022-04-01 to 2025-04-30
Different days in the data: 1125
Area names found in more than one state:
area
Aurangabad    2
Name: state, dtype: int64


 9. Prominent pollutants

In [ ]:
print(raw["prominent_pollutants"].nunique(), "different values")
raw["prominent_pollutants"].value_counts().head(10)

49 different values


,count
prominent_pollutants,
PM10,111053
PM2.5,59670
O3,16202
"PM2.5,PM10",13199
CO,12867
"PM10,O3",3914
SO2,3815
NO2,3012
"PM10,CO",2658


 **Clean the data**

1. Make a working copy

In [ ]:
df = raw.copy()
print(df.shape)

(235785, 9)


2. Drop useless columns

In [ ]:
df = df.drop(columns=["note", "unit"])
print(df.shape)

(235785, 7)


3. Convert date to a real date

In [ ]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)
print(df["date"].dtype)
print(df["date"].min().date(), "to", df["date"].max().date())

datetime64[ns]
2022-04-01 to 2025-04-30


4. Tidy text and create location

In [ ]:
for c in ["state", "area", "prominent_pollutants", "air_quality_status"]:
    df[c] = df[c].str.strip()

df["location"] = df["area"] + " (" + df["state"] + ")"
print(df["location"].nunique(), "unique locations")
df[["area", "state", "location"]].head()

292 unique locations


,area,state,location
0,Amravati,Maharashtra,Amravati (Maharashtra)
1,Purnia,Bihar,Purnia (Bihar)
2,Katni,Madhya Pradesh,Katni (Madhya Pradesh)
3,Tumidih,Chhattisgarh,Tumidih (Chhattisgarh)
4,Byrnihat,Assam,Byrnihat (Assam)


5. Create time columns and season

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["year_month"] = df["date"].dt.to_period("M").astype(str)

def get_season(m):
    if m in (12, 1, 2):
        return "Winter"
    if m in (3, 4, 5):
        return "Summer"
    if m in (6, 7, 8, 9):
        return "Monsoon"
    return "Post-Monsoon"

df["season"] = df["month"].map(get_season)
df[["date", "year", "month", "month_name", "year_month", "season"]].head()

,date,year,month,month_name,year_month,season
0,2025-04-30,2025,4,Apr,2025-04,Summer
1,2025-04-30,2025,4,Apr,2025-04,Summer
2,2025-04-30,2025,4,Apr,2025-04,Summer
3,2025-04-30,2025,4,Apr,2025-04,Summer
4,2025-04-30,2025,4,Apr,2025-04,Summer


6. AQI category and Poor-or-worse flag

In [ ]:
bins = [-1, 50, 100, 200, 300, 400, 10000]
labels = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]

df["aqi_category"] = pd.cut(df["aqi_value"], bins=bins, labels=labels).astype(str)
df["is_poor_or_worse"] = df["aqi_category"].isin(["Poor", "Very Poor", "Severe"]).astype(int)

diff = df[df["aqi_category"] != df["air_quality_status"]]
print("Rows where our category differs from the dataset label:", len(diff))
diff[["date", "location", "aqi_value", "air_quality_status", "aqi_category"]]

Rows where our category differs from the dataset label: 1


,date,location,aqi_value,air_quality_status,aqi_category
126209,2023-11-15,Delhi (Delhi),398,Severe,Very Poor


7. Split prominent pollutants into flag columns

In [ ]:
pollutants = sorted(
    df["prominent_pollutants"].str.split(",").explode().str.strip().unique()
)
print("Pollutants found:", pollutants)

for p in pollutants:
    col = "prom_" + p.replace(".", "").lower()
    df[col] = df["prominent_pollutants"].apply(
        lambda s: int(p in [x.strip() for x in s.split(",")])
    )

df["n_prominent"] = df["prominent_pollutants"].str.count(",") + 1

flag_cols = [c for c in df.columns if c.startswith("prom_")]
df[flag_cols].sum()

Pollutants found: ['CO', 'NH3', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2', 'SO3']


,0
prom_co,18148
prom_nh3,83
prom_no2,6025
prom_o3,24951
prom_pm10,135806
prom_pm25,78030
prom_so2,5284
prom_so3,40


8. Validate the cleaned data

In [ ]:
print("Rows, columns:", df.shape)
print("Missing values:", int(df.isnull().sum().sum()))
print("AQI range:", df["aqi_value"].min(), "to", df["aqi_value"].max())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print()
print(df.columns.tolist())

Rows, columns: (235785, 24)
Missing values: 0
AQI range: 3 to 500
Date range: 2022-04-01 to 2025-04-30

['date', 'state', 'area', 'number_of_monitoring_stations', 'prominent_pollutants', 'aqi_value', 'air_quality_status', 'location', 'year', 'month', 'month_name', 'year_month', 'season', 'aqi_category', 'is_poor_or_worse', 'prom_co', 'prom_nh3', 'prom_no2', 'prom_o3', 'prom_pm10', 'prom_pm25', 'prom_so2', 'prom_so3', 'n_prominent']


9. Save the cleaned dataset

In [ ]:
df = df.sort_values(["date", "state", "area"]).reset_index(drop=True)
df.to_csv("aqi_cleaned.csv", index=False)
print("Saved aqi_cleaned.csv with", df.shape[0], "rows")

from google.colab import files
files.download("aqi_cleaned.csv")

Saved aqi_cleaned.csv with 235785 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>